# WebLooper - Lyrics Processor (Google Colab)

**This notebook runs the heavy AI (lyrics transcription + timing) using your free Colab GPU and writes results back to your Google Drive.**

It integrates perfectly with weblooper's existing Drive sync.

### Quick Steps
1. Runtime → Change runtime type → GPU (T4 recommended, free).
2. Run all cells (it will ask for Drive permission).
3. In the config cell, the `SESSION_FOLDER_ID` is usually already filled (when you clicked 'Run in my Colab' from weblooper it uploaded a ready copy with the ID baked in). If empty, paste the folder ID that weblooper shows you.
4. Cell 1 installs dependencies and **automatically restarts the runtime**. This is expected! After restart, re-run from Cell 2 onward (or just "Run all" again — Cell 1 will detect packages are already installed and skip).
5. In the config cell choose your model (USE_PARAKEET recommended as the best for singing; exactly one must be True). The notebook will find your `vocals.webm`, run the chosen model, and write `lyricTrack.json` + patch `meta.json`.
6. Go back to weblooper and click 'Load results from Colab/Drive' (or reload) — lyrics appear with proper timing!

In [ ]:
# @title 1. Install dependencies (auto-restarts runtime when done)
import subprocess, sys, os

def _is_installed():
    """Check if key packages are importable (skip reinstall on re-run after restart)."""
    try:
        import nemo.collections.asr
        import pydub
        import numpy as np
        # Verify numpy is actually usable (the _center import that fails with stale numpy)
        from numpy._core.umath import _center  # noqa: F401
        return True
    except (ImportError, AttributeError):
        return False

# Skip the entire install if packages are already working (post-restart re-run)
if _is_installed():
    print('Dependencies already installed (post-restart). Skipping to next cell.')
else:
    # --- System dependencies required by NeMo / audio processing ---
    print('Installing system dependencies...')
    subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'sox', 'libsndfile1', 'ffmpeg'],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('System deps: sox, libsndfile1, ffmpeg  \u2713')

    # --- Cython (build dependency for some NeMo sub-packages) ---
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'Cython'])

    # --- Upgrade numba so it accepts numpy 2.x (Colab ships numba 0.60 which caps numpy<2.1) ---
    print('Upgrading numba for numpy 2.x compatibility...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'numba'])

    # --- Core ASR packages ---
    # NeMo >= 2.6.1 supports NumPy 2.x natively. Let pip resolve versions freely.
    print('Installing nemo_toolkit[asr] (this takes 2-4 minutes)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nemo_toolkit[asr]'])

    print('Installing whisperx + utilities...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'whisperx', 'google-auth', 'google-auth-oauthlib',
                           'google-auth-httplib2', 'google-api-python-client', 'pydub'])

    # --- Ensure numpy is consistent: install the version whisperx/nemo agreed on ---
    # After all packages are installed, force numpy to a known-good 2.x version.
    # This resolves the case where Colab's pre-installed numpy 2.0.2 files linger on disk.
    print('Pinning numpy to a consistent version...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           '--force-reinstall', '--no-deps', 'numpy>=2.1,<3'])

    print('\n=== Install complete. Restarting runtime to load new packages... ===')
    print('This is normal! After restart, just re-run all cells (this cell will skip).')

    # Auto-restart the Colab runtime so Python loads the new packages cleanly.
    # os.kill is the most reliable method across Colab runtime versions.
    os.kill(os.getpid(), 9)

In [ ]:
# @title 2. Verify installation
import os

# WhisperX requires this env var on Colab to avoid pyannote weights_only errors
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = 'true'

import numpy as np
import pandas as pd

print('=== INSTALL SUMMARY ===')
print(f'numpy : {np.__version__}')
print(f'pandas: {pd.__version__}')

import nemo.collections.asr as nemo_asr
print('nemo.collections.asr: OK \u2713  (Parakeet path ready)')

try:
    import whisperx
    print(f'whisperx: OK \u2713  (alternative model \u2014 set USE_WHISPERX=True in config to use)')
except ImportError as e:
    print(f'whisperx: NOT AVAILABLE ({e})')
    print('  (Parakeet still works fine \u2014 only set USE_WHISPERX=True if you need it)')

print('=== All good ===')

In [ ]:
# @title 3. Authenticate Google Drive
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload
import io
import json
import os
from pathlib import Path
import time

auth.authenticate_user()
drive_service = build('drive', 'v3')
print('Authenticated to your Google Drive.')

In [ ]:
# @title 4. CONFIGURATION - Paste your session folder ID here
# weblooper will tell you the exact value when you click the button.
# When launched via the weblooper 'Run in my Colab' button, the ID below is pre-filled automatically (no paste needed).
SESSION_FOLDER_ID = "__WEBLOOPER_SESSION_FOLDER_ID__"   # replaced by weblooper on Drive upload (or paste manually)

USE_PARAKEET = True   # Best for song lyrics + speed (recommended)
USE_WHISPERX = False  # Alternative for very accurate word-level timing
# Exactly one of the two above must be True. No silent fallbacks — choose deliberately.

print('Configuration ready.')

In [ ]:
# @title 5. Locate vocal stem in the Drive folder
def list_files_in_folder(folder_id):
    results = drive_service.files().list(
        q=f"'{folder_id}' in parents and trashed=false",
        fields="files(id, name, mimeType)"
    ).execute()
    return results.get('files', [])

def download_file(file_id, dest_path):
    request = drive_service.files().get_media(fileId=file_id)
    with io.FileIO(dest_path, 'wb') as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
    return dest_path

if not SESSION_FOLDER_ID:
    raise RuntimeError("Please paste your SESSION_FOLDER_ID above.")

files = list_files_in_folder(SESSION_FOLDER_ID)
vocal_file = next((f for f in files if 'vocal' in f['name'].lower() and f['name'].endswith(('.webm', '.mp3', '.wav'))), None)

if not vocal_file:
    raise RuntimeError("Could not find a vocals stem in the folder. Make sure the session is uploaded to Drive.")

work_dir = "/content/weblooper_lyrics"
os.makedirs(work_dir, exist_ok=True)
local_vocals = os.path.join(work_dir, vocal_file['name'])
download_file(vocal_file['id'], local_vocals)
print(f"Downloaded vocal stem: {local_vocals}")

OUTPUT_FOLDER_ID = SESSION_FOLDER_ID

In [ ]:
# @title 6. Run the AI model (Parakeet or WhisperX)
from pydub import AudioSegment

audio = AudioSegment.from_file(local_vocals)
duration_sec = len(audio) / 1000.0
print(f"Processing audio of length {duration_sec:.1f} seconds...")

# Convert to mono 16kHz WAV (models expect single-channel input;
# stereo vocals.webm causes shape mismatch errors in Parakeet)
local_vocals_mono = os.path.join(work_dir, 'vocals_mono.wav')
audio.set_channels(1).set_frame_rate(16000).export(local_vocals_mono, format='wav')
print(f"Converted to mono 16kHz WAV for model input.")

if USE_PARAKEET:
    print("Loading Parakeet TDT 0.6B (excellent for song lyrics)...")
    import nemo.collections.asr as nemo_asr
    model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v2")
    result = model.transcribe([local_vocals_mono], timestamps=True)[0]
    text = result.text
    chunks = []
    if hasattr(result, 'timestamp') and result.timestamp:
        for w in result.timestamp.get('word', []):
            chunks.append({"text": w['word'], "timestamp": [w['start'], w['end']]})
    else:
        words = text.split()
        for i, w in enumerate(words):
            s = (i / max(1, len(words))) * duration_sec
            e = ((i + 1) / max(1, len(words))) * duration_sec
            chunks.append({"text": w, "timestamp": [s, e]})
    print("Parakeet finished.")

elif USE_WHISPERX:
    print("Loading WhisperX (best word-level timing)...")
    import whisperx
    device = "cuda"
    model = whisperx.load_model("large-v3", device, compute_type="float16")
    audio_wav = whisperx.load_audio(local_vocals_mono)
    result = model.transcribe(audio_wav, batch_size=16)
    model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
    result = whisperx.align(result["segments"], model_a, metadata, audio_wav, device)
    text = " ".join([seg["text"] for seg in result["segments"]])
    chunks = []
    for seg in result["segments"]:
        for word in seg.get("words", []):
            chunks.append({"text": word["word"], "timestamp": [word["start"], word["end"]]})
    print("WhisperX finished.")

else:
    raise RuntimeError("Set exactly one of USE_PARAKEET=True or USE_WHISPERX=True in the CONFIG cell.")

# Clean up the large temporary WAV file (can be 50-100MB+)
os.remove(local_vocals_mono)
print("Cleaned up temporary mono WAV.")

print("=== RAW TEXT (first 400 chars) ===")
print(text[:400] + "..." if len(text) > 400 else text)

In [ ]:
# @title 7. Build LyricTrack (weblooper format) — gap-based line splitting from word timestamps

# --- Gap-based line splitting ---
# Song transcription models (Parakeet, WhisperX) often return text WITHOUT
# punctuation. Splitting on punctuation alone results in one giant block.
# Instead, we detect natural phrase boundaries by looking at silence gaps
# between consecutive words in the timestamp data.

GAP_THRESHOLD = 0.35   # seconds of silence between words to trigger a new line
MAX_WORDS_PER_LINE = 12  # safety cap even if no gap detected

has_real_timestamps = len(chunks) > 0 and chunks[0].get('timestamp', [0, 0])[1] > 0

segments = []

if has_real_timestamps:
    # Build lines by detecting gaps between words
    current_line_chunks = [chunks[0]]

    for i in range(1, len(chunks)):
        prev_end = chunks[i - 1]['timestamp'][1]
        curr_start = chunks[i]['timestamp'][0]
        gap = curr_start - prev_end

        # Start a new line if there's a significant gap or we hit the word cap
        if gap >= GAP_THRESHOLD or len(current_line_chunks) >= MAX_WORDS_PER_LINE:
            # Flush current line
            line_text = ' '.join(c['text'] for c in current_line_chunks)
            start = round(current_line_chunks[0]['timestamp'][0], 3)
            end = round(current_line_chunks[-1]['timestamp'][1], 3)
            segments.append({
                "id": f"colab_{int(time.time())}_{len(segments)}",
                "start": start,
                "end": end,
                "text": line_text,
                "source": "ai",
                "model": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3"
            })
            current_line_chunks = []

        current_line_chunks.append(chunks[i])

    # Flush the last line
    if current_line_chunks:
        line_text = ' '.join(c['text'] for c in current_line_chunks)
        start = round(current_line_chunks[0]['timestamp'][0], 3)
        end = round(current_line_chunks[-1]['timestamp'][1], 3)
        segments.append({
            "id": f"colab_{int(time.time())}_{len(segments)}",
            "start": start,
            "end": end,
            "text": line_text,
            "source": "ai",
            "model": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3"
        })

    print(f"Built {len(segments)} segments using gap-based splitting (threshold={GAP_THRESHOLD}s, max_words={MAX_WORDS_PER_LINE}).")

else:
    # Fallback: uniform distribution (only if model returned no timestamps)
    import re
    def smart_split_lyrics(txt, max_chars=75):
        parts = re.split(r'([.!?\u3002\uff01\uff1f\n])', txt)
        lines = []
        current = ""
        for p in parts:
            current += p
            if len(current.strip()) > max_chars or p in '.!?\u3002\uff01\uff1f\n':
                if current.strip():
                    lines.append(current.strip())
                current = ""
        if current.strip():
            lines.append(current.strip())
        return [l for l in lines if l]

    lines = smart_split_lyrics(text)
    for i, line in enumerate(lines):
        start = round((i / max(1, len(lines))) * duration_sec, 3)
        end = round(((i + 1) / max(1, len(lines))) * duration_sec, 3)
        segments.append({
            "id": f"colab_{int(time.time())}_{i}",
            "start": start,
            "end": end,
            "text": line,
            "source": "ai",
            "model": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3"
        })
    print(f"Built {len(segments)} segments using uniform timing (no word timestamps available).")

lyric_track = {
    "id": f"lt_colab_{int(duration_sec)}",
    "stemSessionId": SESSION_FOLDER_ID,
    "duration": duration_sec,
    "segments": segments,
    "metadata": {
        "generatedAt": int(time.time() * 1000),
        "lyricsModel": "parakeet-tdt-0.6b-v2" if USE_PARAKEET else "whisperx-large-v3",
        "vocalsStemUsed": True,
        "source": "google-colab-free-gpu"
    },
    "version": 1
}

local_path = "/content/lyricTrack.json"
with open(local_path, "w") as f:
    json.dump(lyric_track, f, indent=2)

print(f"\nLyricTrack: {len(segments)} segments, duration {duration_sec:.1f}s")
print("=== TIMING PREVIEW (first 8 segments) ===")
for s in segments[:8]:
    print(f"  {s['start']:6.1f}s - {s['end']:6.1f}s : {s['text'][:60]}")

In [ ]:
# @title 8. Write back to your Drive folder (lyricTrack.json + patch meta.json)
from googleapiclient.http import MediaIoBaseUpload

if OUTPUT_FOLDER_ID:
    # Upload sidecar
    media = MediaIoBaseUpload(open(local_path, "rb"), mimetype="application/json")
    drive_service.files().create(
        body={"name": "lyricTrack.json", "parents": [OUTPUT_FOLDER_ID]},
        media_body=media
    ).execute()
    print("Uploaded lyricTrack.json")

    # Patch meta.json so weblooper picks it up automatically
    try:
        meta_files = drive_service.files().list(
            q=f"'{OUTPUT_FOLDER_ID}' in parents and name='meta.json' and trashed=false",
            fields="files(id)"
        ).execute().get('files', [])
        if meta_files:
            meta_id = meta_files[0]['id']
            req = drive_service.files().get_media(fileId=meta_id)
            meta_bytes = io.BytesIO()
            downloader = MediaIoBaseDownload(meta_bytes, req)
            done = False
            while not done:
                _, done = downloader.next_chunk()
            current_meta = json.loads(meta_bytes.getvalue().decode('utf-8'))
            current_meta['lyricTrack'] = lyric_track
            media = MediaIoBaseUpload(
                io.BytesIO(json.dumps(current_meta, indent=2).encode('utf-8')),
                mimetype='application/json'
            )
            drive_service.files().update(fileId=meta_id, media_body=media).execute()
            print("Patched meta.json with lyricTrack \u2014 weblooper will see it on reload!")
    except Exception as e:
        print(f"Could not patch meta.json: {e}")

    print("\n=== SUCCESS ===")
    print("Go back to weblooper and reload this stem session. The lyrics should now have proper timing.")
else:
    print("No folder ID \u2014 please download the file manually and place it in your session folder.")
    from google.colab import files
    files.download(local_path)

**That's it!** 

The results are written back to the exact same Drive folder weblooper uses. 
Reload the session in the app and the timed lyrics will appear.